In [32]:
pip install tenseal

In [41]:
import torch
from torchvision import datasets
import torchvision.transforms as transforms
import numpy as np
from torch.utils.data import Subset
from sklearn.metrics import classification_report
from tqdm import tqdm
torch.manual_seed(1)


In [34]:
# Load full datasets
train_data = datasets.MNIST('data', train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.MNIST('data', train=False, download=True, transform=transforms.ToTensor())

# Print dataset sizes
print(f"Total training samples: {len(train_data)}")
print(f"Total testing samples: {len(test_data)}")

# Define percentage splits
train_split = 1
test_split = 0.1

# Calculate subset indices
train_size = int(train_split * len(train_data))
test_size = int(test_split * len(test_data))

train_indices = np.random.choice(len(train_data), train_size, replace=False)
test_indices = np.random.choice(len(test_data), test_size, replace=False)

# Create subset datasets
train_subset = Subset(train_data, train_indices)
test_subset = Subset(test_data, test_indices)
# Define batch size
batch_size = 64

# Create data loaders using subsets
train_loader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_subset, batch_size=batch_size, shuffle=True)


Total training samples: 60000
Total testing samples: 10000


In [35]:
class ConvNet(torch.nn.Module):
    def __init__(self, hidden_dim=64, num_classes=10):
        super().__init__()
        self.conv = torch.nn.Conv2d(
            in_channels=1,
            out_channels=4,
            kernel_size=7,
            stride=3,
            padding=0
        )
        self.fc_hidden = torch.nn.Linear(256, hidden_dim)
        self.fc_out = torch.nn.Linear(hidden_dim, num_classes)

    def _approx_relu(self, x):
        # polynomial approximation of ReLU
        return x * x + x

    def forward(self, x):
        x = self.conv(x)
        x = self._approx_relu(x)

        # flatten features, keep batch dimension
        x = x.reshape(x.size(0), -1)

        x = self.fc_hidden(x)
        x = self._approx_relu(x)
        x = self.fc_out(x)
        return x


def train(model, dataloader, loss_fn, optim, epochs=10):
    model.train()

    for ep in range(1, epochs + 1):
        running_loss = 0.0

        for batch_x, batch_y in dataloader:
            optim.zero_grad()
            preds = model(batch_x)
            loss = loss_fn(preds, batch_y)
            loss.backward()
            optim.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(dataloader)
        print(f"Epoch {ep} | Training Loss: {avg_loss:.6f}")

    model.eval()
    return model


In [36]:
net = ConvNet(hidden_dim=64, num_classes=10)

loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.Adam(net.parameters(), lr=0.01)

net = train(
    model=net,
    dataloader=train_loader,
    loss_fn=loss_fn,
    optim=opt,
    epochs=10
)


Epoch 1 | Training Loss: 0.234046
Epoch 2 | Training Loss: 0.152281
Epoch 3 | Training Loss: 0.150429
Epoch 4 | Training Loss: 0.142348
Epoch 5 | Training Loss: 0.150614
Epoch 6 | Training Loss: 0.140228
Epoch 7 | Training Loss: 0.146228
Epoch 8 | Training Loss: 0.160265
Epoch 9 | Training Loss: 0.133369
Epoch 10 | Training Loss: 0.143654


In [37]:
frac = 0.1
num_eval = int(len(test_data) * frac)

perm = torch.randperm(len(test_data))[:num_eval]
eval_set = Subset(test_data, perm)

test_loader = DataLoader(eval_set, batch_size=1, shuffle=False)

print(f"Evaluating on {num_eval} test samples ({int(frac * 100)}%)")


Evaluating on 1000 test samples (10%)


In [40]:
def evaluate(net, loader, loss_fn, n_classes=10):
    loss_sum = 0.0
    correct_per_class = [0] * n_classes
    count_per_class = [0] * n_classes

    preds = []
    targets = []

    net.eval()
    with torch.no_grad():
        for x, y in loader:
            logits = net(x)
            loss = loss_fn(logits, y)
            loss_sum += loss.item()

            predicted = logits.argmax(dim=1)

            true_label = y.item()
            pred_label = predicted.item()

            count_per_class[true_label] += 1
            if pred_label == true_label:
                correct_per_class[true_label] += 1

            preds.append(pred_label)
            targets.append(true_label)

    avg_loss = loss_sum / len(loader)
    print(f"Test Loss: {avg_loss:.6f}\n")

    for c in range(n_classes):
        if count_per_class[c] == 0:
            acc = 0.0
        else:
            acc = 100.0 * correct_per_class[c] / count_per_class[c]

        print(
            f"Test Accuracy of class {c}: "
            f"{acc:.0f}% ({correct_per_class[c]}/{count_per_class[c]})"
        )

    total_correct = sum(correct_per_class)
    total_seen = sum(count_per_class)
    overall_acc = 100.0 * total_correct / total_seen

    print(
        f"\nTest Accuracy (Overall): "
        f"{overall_acc:.0f}% ({total_correct}/{total_seen})"
    )

    print("\nClassification Report:")
    print(classification_report(targets, preds, digits=4))


evaluate(net, test_loader, loss_fn)


Test Loss: 0.187354

Test Accuracy of class 0: 100% (88/88)
Test Accuracy of class 1: 95% (121/127)
Test Accuracy of class 2: 86% (85/99)
Test Accuracy of class 3: 95% (97/102)
Test Accuracy of class 4: 99% (95/96)
Test Accuracy of class 5: 98% (89/91)
Test Accuracy of class 6: 98% (89/91)
Test Accuracy of class 7: 97% (102/105)
Test Accuracy of class 8: 98% (93/95)
Test Accuracy of class 9: 96% (102/106)

Test Accuracy (Overall): 96% (961/1000)

Classification Report:
              precision    recall  f1-score   support

           0     0.9670    1.0000    0.9832        88
           1     1.0000    0.9528    0.9758       127
           2     0.9770    0.8586    0.9140        99
           3     0.9417    0.9510    0.9463       102
           4     0.9596    0.9896    0.9744        96
           5     0.9570    0.9780    0.9674        91
           6     0.9889    0.9780    0.9834        91
           7     0.9714    0.9714    0.9714       105
           8     0.9118    0.9789    0.

In [52]:
class EncryptedConvNet:
    def __init__(self, torch_model):
        # Automatically detect convolution layers
        conv_layers = [m for m in torch_model.modules() if isinstance(m, torch.nn.Conv2d)]
        conv = conv_layers[0]
        self.conv_weights = [
            conv.weight.data[i].view(*conv.kernel_size).tolist()
            for i in range(conv.out_channels)
        ]
        self.conv_biases = conv.bias.data.tolist()

        # Automatically detect linear layers
        linear_layers = [m for m in torch_model.modules() if isinstance(m, torch.nn.Linear)]
        self.fc1_weights = linear_layers[0].weight.T.data.tolist()
        self.fc1_biases = linear_layers[0].bias.data.tolist()
        self.fc2_weights = linear_layers[1].weight.T.data.tolist()
        self.fc2_biases = linear_layers[1].bias.data.tolist()

    def forward(self, enc_x, windows_nb):
        # Encrypted convolution
        enc_channels = [
            enc_x.conv2d_im2col(k, windows_nb) + b
            for k, b in zip(self.conv_weights, self.conv_biases)
        ]
        enc_x = ts.CKKSVector.pack_vectors(enc_channels)
        enc_x += enc_x.square()  # approximate ReLU

        # Encrypted fully connected layers
        enc_x = enc_x.mm(self.fc1_weights) + self.fc1_biases
        enc_x += enc_x.square()
        enc_x = enc_x.mm(self.fc2_weights) + self.fc2_biases
        return enc_x

    def __call__(self, enc_x, windows_nb):
        return self.forward(enc_x, windows_nb)


def encrypted_test(context, enc_model, loader, loss_fn, kernel_shape, stride):
    total_loss = 0.0
    correct_per_class = [0.0] * 10
    count_per_class = [0.0] * 10
    all_preds, all_labels = [], []

    for data, target in tqdm(loader, desc="Encrypted Evaluation"):
        # Encode input using im2col
        enc_x, windows_nb = ts.im2col_encoding(
            context,
            data.view(28, 28).tolist(),
            kernel_shape[0], kernel_shape[1],
            stride
        )

        # Forward pass (encrypted)
        enc_out = enc_model(enc_x, windows_nb)  # now works thanks to __call__
        decrypted = torch.tensor(enc_out.decrypt()).view(1, -1)

        # Compute loss
        loss = loss_fn(decrypted, target)
        total_loss += loss.item()

        # Predictions
        pred = decrypted.argmax(dim=1)
        label = target.item()
        is_correct = pred.item() == label

        correct_per_class[label] += int(is_correct)
        count_per_class[label] += 1
        all_preds.append(pred.item())
        all_labels.append(label)

    # Average loss
    avg_loss = total_loss / sum(count_per_class)
    print(f"Test Loss: {avg_loss:.6f}\n")

    # Class-wise accuracy
    for c in range(10):
        acc = (100 * correct_per_class[c] / count_per_class[c]) if count_per_class[c] else 0
        print(f"Accuracy for class {c}: {int(acc)}% "
              f"({int(correct_per_class[c])}/{int(count_per_class[c])})")

    # Overall accuracy
    total_correct = sum(correct_per_class)
    total_samples = sum(count_per_class)
    overall_acc = 100 * total_correct / total_samples
    print(f"\nOverall Test Accuracy: {int(overall_acc)}% "
          f"({int(total_correct)}/{int(total_samples)})")

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, digits=4))


# --- Encryption context setup ---
scale_bits = 26
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[31] + [scale_bits]*6 + [31]
)
context.global_scale = 2 ** scale_bits
context.generate_galois_keys()

# Get kernel/stride from current ConvNet
kernel_shape = net.conv.kernel_size
stride = net.conv.stride[0]

# --- Example usage ---
enc_model = EncryptedConvNet(net)
# encrypted_test(context, enc_model, test_loader, loss_fn, kernel_shape, stride)


In [53]:
# Convert to encrypted model
enc_model = EncryptedConvNet(net)  # updated class name and variable

# Get conv layer info from the new model
kernel_shape = net.conv.kernel_size
stride = net.conv.stride[0]

# Run encrypted test
encrypted_test(
    context,
    enc_model,
    test_loader,
    loss_fn,
    kernel_shape,
    stride
)


Encrypted Evaluation: 100%|██████████| 1000/1000 [1:09:21<00:00,  4.16s/it]

Test Loss: 0.245169

Accuracy for class 0: 100% (88/88)
Accuracy for class 1: 95% (121/127)
Accuracy for class 2: 81% (81/99)
Accuracy for class 3: 94% (96/102)
Accuracy for class 4: 98% (95/96)
Accuracy for class 5: 95% (87/91)
Accuracy for class 6: 95% (87/91)
Accuracy for class 7: 96% (101/105)
Accuracy for class 8: 97% (93/95)
Accuracy for class 9: 96% (102/106)

Overall Test Accuracy: 95% (951/1000)

Classification Report:
              precision    recall  f1-score   support

           0     0.9670    1.0000    0.9832        88
           1     1.0000    0.9528    0.9758       127
           2     0.9878    0.8182    0.8950        99
           3     0.9600    0.9412    0.9505       102
           4     0.9596    0.9896    0.9744        96
           5     0.9667    0.9560    0.9613        91
           6     0.9886    0.9560    0.9721        91
           7     0.9712    0.9619    0.9665       105
           8     0.8087    0.9789    0.8857        95
           9     0.9273    